# Becruily & Frazer BS-Roformer Karaoke

This notebook runs the exact Becruily & Frazer BS-Roformer Karaoke checkpoint as a separate stem separation. It does not load or modify the RIFT-SVC model.

Add one or more audio files as a Kaggle Dataset input, enable Internet, then run all cells. The notebook writes the `Vocals` and `Instrumental` stems as Float WAV files under `/kaggle/working/outputs/`.

Model source: `https://huggingface.co/becruily/bs-roformer-karaoke`, checkpoint `bs_roformer_karaoke_frazer_becruily.ckpt`. The public checkpoint is downloaded at runtime and is not stored in this repository.

In [ ]:
%pip install -q "ml-collections>=0.1.1" "omegaconf>=2.2.3" "rotary-embedding-torch==0.3.5" "einops>=0.8.1" "beartype>=0.14" "librosa>=0.10" "soundfile>=0.12" "tqdm>=4.67" "matplotlib>=3.8" "huggingface-hub>=0.23"

In [ ]:
from pathlib import Path
import subprocess
import sys

MSST_DIR = Path('/kaggle/working/msst-becruily-frazer')
MSST_REPO = 'https://github.com/ZFTurbo/Music-Source-Separation-Training.git'
MSST_COMMIT = 'e247dfe4abc1f17c69dff719207fe045dc04413a'

if MSST_DIR.exists() and not (MSST_DIR / '.git').exists():
    import shutil
    shutil.rmtree(MSST_DIR)
if not (MSST_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', MSST_REPO, str(MSST_DIR)], check=True)
subprocess.run(['git', '-C', str(MSST_DIR), 'fetch', '--depth', '1', 'origin', MSST_COMMIT], check=True)
subprocess.run(['git', '-C', str(MSST_DIR), 'checkout', MSST_COMMIT], check=True)
sys.path.insert(0, str(MSST_DIR))
print('MSST revision:', subprocess.check_output(['git', '-C', str(MSST_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = 'becruily/bs-roformer-karaoke'
MODEL_REVISION = 'f7849ae934209184dc288d1018cd8a76a7fc8b3c'
MODEL_DIR = Path('/kaggle/working/becruily-frazer-model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename='bs_roformer_karaoke_frazer_becruily.ckpt',
    revision=MODEL_REVISION,
    local_dir=str(MODEL_DIR),
))
CONFIG_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename='config_karaoke_frazer_becruily.yaml',
    revision=MODEL_REVISION,
    local_dir=str(MODEL_DIR),
))

if MODEL_PATH.stat().st_size < 100 * 1024 * 1024:
    raise RuntimeError(f'Checkpoint looks incomplete: {MODEL_PATH.stat().st_size} bytes')
print(f'checkpoint: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024**2:.1f} MiB)')
print(f'config: {CONFIG_PATH}')

In [ ]:
from shutil import rmtree

AUDIO_EXTENSIONS = {'.wav', '.flac', '.m4a', '.mp3', '.ogg', '.opus'}
MANUAL_INPUT = None  # Example: '/kaggle/input/my-dataset/song.wav'
INPUT_DIR = Path('/kaggle/working/becruily-frazer-input')

if INPUT_DIR.exists():
    rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True)

if MANUAL_INPUT:
    candidates = [Path(MANUAL_INPUT)]
else:
    candidates = sorted(
        p for p in Path('/kaggle/input').rglob('*')
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
    )

if not candidates or any(not p.exists() for p in candidates):
    raise FileNotFoundError('No valid audio input was found. Add a Kaggle Dataset or set MANUAL_INPUT.')

for index, source in enumerate(candidates):
    target = INPUT_DIR / f'{index:02d}-{source.name}'
    target.symlink_to(source)
    print(target, '<-', source)
print(f'{len(candidates)} input file(s) selected')

## Inference settings

`batch_size=1` is used for T4 stability. The model config uses a 20-second window with four-way overlap. Set `USE_TTA = True` for a slower extra candidate if the first result needs comparison; it runs additional polarity/channel passes.

In [ ]:
from types import SimpleNamespace
import torch

from inference import run_folder
from utils.model_utils import load_start_checkpoint
from utils.settings import get_model_from_config

OUTPUT_DIR = Path('/kaggle/working/outputs/becruily-frazer-bs-roformer-karaoke')
if OUTPUT_DIR.exists():
    rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
USE_TTA = False
BIGSHIFTS = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

model, config = get_model_from_config('bs_roformer', str(CONFIG_PATH))
config.inference.batch_size = 1

try:
    checkpoint = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)
except TypeError:
    checkpoint = torch.load(str(MODEL_PATH), map_location='cpu')

args = SimpleNamespace(
    start_check_point=str(MODEL_PATH),
    model_type='bs_roformer',
    load_only_compatible_weights=False,
    lora_checkpoint_loralib='',
    input_folder=str(INPUT_DIR),
    store_dir=str(OUTPUT_DIR),
    extract_instrumental=True,
    disable_detailed_pbar=False,
    force_cpu=(device.type != 'cuda'),
    pcm_type='FLOAT',
    flac_file=False,
    use_tta=USE_TTA,
    bigshifts=BIGSHIFTS,
    filename_template='{file_name}__becruily-frazer-bs-roformer-karaoke_{instr}',
    draw_spectro=0,
)

load_start_checkpoint(args, model, checkpoint, type_='inference')
model = model.to(device).eval()
run_folder(model, args, config, device, verbose=True)

In [ ]:
import shutil
import soundfile as sf

wav_outputs = sorted(OUTPUT_DIR.rglob('*.wav'))
if not wav_outputs:
    raise RuntimeError('Inference completed without producing WAV output')

for path in wav_outputs:
    info = sf.info(path)
    print(f'{path} | {info.samplerate} Hz | {info.channels} ch | {info.subtype} | {info.duration:.2f}s')

archive = shutil.make_archive(
    '/kaggle/working/becruily-frazer-bs-roformer-karaoke-wav',
    'zip',
    root_dir=OUTPUT_DIR,
)
print('download archive:', archive)

from IPython.display import FileLink
FileLink('becruily-frazer-bs-roformer-karaoke-wav.zip')

### 临时外链下载（可选）

下一单元会把 ZIP 上传到临时文件服务并打印公开下载链接。链接只用于临时下载，下载完成后即可失效或过期。

In [ ]:
from pathlib import Path
import subprocess

archive_path = Path('/kaggle/working/becruily-frazer-bs-roformer-karaoke-wav.zip')
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise FileNotFoundError(f'Archive is missing or empty: {archive_path}')

import json

def parse_upload_response(service, response):
    lines = [line.strip() for line in response.splitlines() if line.strip()]
    if lines and lines[-1].startswith(('http://', 'https://')):
        return lines[-1]
    try:
        payload = json.loads(response)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{service} returned an unexpected response: {response!r}') from exc

    if service == 'tmpfiles.org':
        url = payload.get('data', {}).get('url')
        if url and '://tmpfiles.org/' in url:
            return url.replace('://tmpfiles.org/', '://tmpfiles.org/dl/', 1)
    for key in ('link', 'directLink', 'url'):
        url = payload.get(key)
        if isinstance(url, str) and url.startswith(('http://', 'https://')):
            return url
    raise RuntimeError(f'{service} returned no download URL: {response!r}')

def run_upload(service, command):
    result = subprocess.run(command, capture_output=True, text=True, timeout=1800)
    if result.returncode != 0:
        detail = (result.stderr or result.stdout).strip()
        raise RuntimeError(detail or f'upload exited with {result.returncode}')
    return parse_upload_response(service, result.stdout.strip())

curl_options = [
    'curl', '--fail', '--silent', '--show-error', '--retry', '2',
    '--retry-delay', '3', '--connect-timeout', '20', '--max-time', '1800',
]
uploaders = [
    ('litterbox.catbox.moe', curl_options + [
        '-F', 'reqtype=fileupload', '-F', 'time=72h',
        '-F', f'fileToUpload=@{archive_path}',
        'https://litterbox.catbox.moe/resources/internals/api.php',
    ]),
    ('catbox.moe', curl_options + [
        '-F', 'reqtype=fileupload', '-F', f'fileToUpload=@{archive_path}',
        'https://catbox.moe/user/api.php',
    ]),
    ('tmpfiles.org', curl_options + [
        '-F', f'file=@{archive_path}', 'https://tmpfiles.org/api/v1/upload',
    ]),
    ('file.io', curl_options + [
        '-F', f'file=@{archive_path}', 'https://file.io',
    ]),
    ('0x0.st', curl_options + [
        '-F', f'file=@{archive_path}', 'https://0x0.st',
    ]),
    ('transfer.sh', curl_options + [
        '--upload-file', str(archive_path),
        f'https://transfer.sh/{archive_path.name}',
    ]),
]

for service, command in uploaders:
    try:
        temporary_url = run_upload(service, command)
        print(f'{service} temporary download URL (expires according to service policy):')
        print(temporary_url)
        break
    except Exception as exc:
        print(f'{service} upload failed: {exc}')
else:
    print('All temporary upload services failed; use the FileLink above or retry later.')